# Imports and initialized some clients

In [1]:
import os
import re
import string
import numpy as np
import pandas as pd
import faiss
import json
import time
import requests


from tqdm import tqdm
tqdm.pandas()

import nltk
# nltk.download("punkt", quiet=True)
# nltk.download("stopwords", quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords as nltk_stopwords

from rank_bm25 import BM25Okapi
from sklearn.preprocessing import minmax_scale

from openai import OpenAI

from dotenv import load_dotenv
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

import cohere
co = cohere.Client(os.getenv("COHERE_API_KEY"))

# Configurations

In [2]:
EMBED_MODEL = "text-embedding-3-small"   # cheap & solid
GEN_MODEL = "gpt-4o-mini"                # choose a chat model you have access to
FAISS_MODE = "cosine"                    # "cosine" or "similarity_score"
ALPHA = 0.8                              # semantic weight in fusion
TOP_K = 5                                # top docs to return after fusion
RERANK_TOP_K = 5                         # top docs after reranking

STOPWORDS = set(nltk_stopwords.words("english"))

# Utils

In [3]:
def preprocess_text(text: str):
    """Clean and normalize text by removing extra whitespace and fixing spacing around punctuation."""
    if pd.isna(text):
        return None
    text = str(text)
    if (text.startswith('"') and text.endswith('"')) or (text.startswith("'") and text.endswith("'")):
        text = text[1:-1].strip()
    text = re.sub(r'\s+', ' ', text)                    # collapse whitespace
    text = re.sub(r'([.,?!])\s*', r'\1 ', text)         # ensure space after punctuation
    text = re.sub(r'\s+([.,?!])', r'\1', text)          # remove space before punctuation
    return text.strip()


def embed_texts(texts, model: str = EMBED_MODEL, batch_size: int = 64):
    """Embed a list of texts in batches with progress bar."""
    embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding batches"):
        batch = texts[i:i+batch_size]
        # Safety truncation for extreme inputs
        batch = [t[:10000] if len(t) > 10000 else t for t in batch]
        resp = client.embeddings.create(input=batch, model=model)
        batch_embeddings = [r.embedding for r in resp.data]
        embeddings.extend(batch_embeddings)
    return np.array(embeddings)


def build_faiss_index(df: pd.DataFrame, mode: str, save_path="sanity_llm_index.faiss"):
    emb = np.array(df["embedding"].tolist()).astype(np.float32)
    dim = emb.shape[1]

    # Map FAISS row index -> (Context, Response)
    id_to_meta = {i: {"Context": row["Context"], "Response": row["Response"]} 
                  for i, row in df.iterrows()}

    if mode == "cosine":
        faiss.normalize_L2(emb)
        index = faiss.IndexFlatIP(dim)
    elif mode == "similarity_score":
        index = faiss.IndexFlatL2(dim)
    else:
        raise ValueError("mode must be either 'cosine' or 'similarity_score'")

    index.add(emb)
    faiss.write_index(index, save_path)

    return index, id_to_meta


def load_faiss_db(df: pd.DataFrame, mode: str, save_path="sanity_llm_index.faiss"):
    try:
        index = faiss.read_index(save_path)
        id_to_meta = {i: {"Context": row["Context"], "Response": row["Response"]} 
                      for i, row in df.iterrows()}
    except:
        index, id_to_meta = build_faiss_index(df, mode, save_path)

    return index, id_to_meta

def GetSemanticScores(query: str, index, mode="cosine"):
    q_emb = np.array([get_embedding(query)], dtype=np.float32)
    if mode == "cosine":
        faiss.normalize_L2(q_emb)
        D, I = index.search(q_emb, k=index.ntotal)
        scores = D[0]  # higher is better
    else:
        D, I = index.search(q_emb, k=index.ntotal)
        scores = 1.0 / (D[0] + 1e-6)  # convert L2 to similarity
    return {int(I[0][i]): float(scores[i]) for i in range(len(I[0]))}


def GetBM25Score(query: str, bm25, stop_set):
    tokens = [t for t in word_tokenize(query.lower()) if t not in stop_set and t not in string.punctuation]
    scores = bm25.get_scores(tokens)  # aligned with corpus order
    return {i: float(s) for i, s in enumerate(scores)}


def FuseScores(query: str, bm25, index, mode="cosine", alpha=ALPHA, stop_set=STOPWORDS):
    bm25_scores = GetBM25Score(query, bm25, stop_set)
    semantic_scores = GetSemanticScores(query, index, mode=mode)

    doc_ids = list(bm25_scores.keys())
    bm25_vals = np.array([bm25_scores[doc_id] for doc_id in doc_ids])
    sem_vals = np.array([semantic_scores.get(doc_id, 0.0) for doc_id in doc_ids])

    # normalize [0,1]
    bm25_norm = minmax_scale(bm25_vals) if len(bm25_vals) > 1 else bm25_vals
    sem_norm = minmax_scale(sem_vals) if len(sem_vals) > 1 else sem_vals

    fused = alpha * sem_norm + (1 - alpha) * bm25_norm
    return doc_ids, fused


def generate_stepback_question(user_query: str):
    prompt = f"""
You are a helpful mental health assistant. Given a user's emotional/mental health concern,
write ONE insightful "stepback question" that helps the user reflect at a higher level.
User: {user_query}
Your single stepback question:""".strip()

    # Keep it a single-turn prompt
    resp = client.chat.completions.create(
        model=GEN_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=64
    )
    return resp.choices[0].message.content.strip()


def rerank_with_cohere(query: str, docs: list, top_k: int = RERANK_TOP_K):
    """Rerank a list of dicts [{'Context', 'Response'}] by scoring on their Context text, and return the dicts sorted."""
    if not docs:
        return []

    texts = [f"Context: {d['Context']}\nResponse: {d['Response']}" for d in docs]  # lightweight reranking on question only
    response = co.rerank(
        model="rerank-v3.5",
        query=query,
        documents=texts,
        top_n=min(top_k, len(texts)),
    )
    # Map back to dicts in reranked order
    reranked = [docs[r.index] for r in response.results]
    return reranked



def build_generation_prompt(user_query: str, docs: list):
    """Formats the prompt for generation using retrieved contexts (Context + bulleted Responses)."""
    blocks = []
    for i, d in enumerate(docs, 1):
        blocks.append(f"Document {i}:\nQuestion: {d['Context']}\nAnswers:\n{d['Response']}")
    context_section = "\n\n".join(blocks)

    prompt = f"""You are a helpful and empathetic mental health assistant.
Use the provided Q&A knowledge to answer the user's question in a thoughtful, safe, and non-diagnostic way.
Cite insights from the answers implicitly (no numbers/links), and include supportive next steps (e.g., grounded coping strategies).
If urgent risk is implied, advise seeking immediate professional help or contacting local emergency services.

NOTE: Your Task is only to respond to the User Query related to mental health support. 
If the user query is not related to mental health support, respond with:
"I'm here to help with mental health support. This topic seems outside that scope."
ANOTHER NOTE: You only able to output maximum 512 tokens. Please do not exceed this limit.

--- Knowledge ---
{context_section}

--- User Query ---
{user_query}

--- Your Answer ---
"""
    return prompt


def generate_single_response(user_query, client, final_docs, model="gpt-4o-mini"):
    """
    Generate a single response without conversation history.
    - user_query: raw user question
    - client: OpenAI client instance
    - final_docs: retrieved documents
    """
    prompt = build_generation_prompt(user_query, final_docs)

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=512
    )

    assistant_reply = response.choices[0].message.content.strip()
    return assistant_reply


def get_embedding(text, model="text-embedding-3-small"):
    response = client.embeddings.create(
        input=[text],
        model=model
    )
    return response.data[0].embedding

# Load & preprocess data just once

In [17]:
raw1 = pd.read_json("data.json")
raw1.drop(columns=["instruction"], inplace=True, errors="ignore")
raw1.rename(columns={"input": "Context", "output": "Response"}, inplace=True)

# Load Marmikpandya dataset
raw2 = pd.read_json("combined_dataset.json", lines=True)

# Normalize schema (in case names differ, adjust if needed)
if "Context" not in raw2.columns or "Response" not in raw2.columns:
    # Example mapping if needed (adjust per schema of marmikpandya dataset)
    if "question" in raw2.columns and "answer" in raw2.columns:
        raw2.rename(columns={"question": "Context", "answer": "Response"}, inplace=True)

raw1["Context"] = raw1["Context"].apply(preprocess_text)
raw1["Response"] = raw1["Response"].apply(preprocess_text)

raw2["Context"] = raw2["Context"].apply(preprocess_text)
raw2["Response"] = raw2["Response"].apply(preprocess_text)

# Combine datasets
raw = pd.concat([raw1, raw2], ignore_index=True)

# Drop null/short responses
raw = raw.dropna(subset=["Response"])
raw = raw[raw["Response"].str.strip().str.len() >= 5]

# Drop duplicate (Context, Response) pairs only (keep same Context with multiple Responses)
raw = raw.drop_duplicates(subset=["Context", "Response"])

# Group: Context -> bulleted Responses
df_grouped = (
    raw.groupby("Context")["Response"]
       .apply(lambda xs: "\n".join(f"- {x}" for x in xs))
       .reset_index()
)

df_grouped["doc_id"] = df_grouped.index.astype(str)
# Save outputs
# raw.to_csv("CombinedRaw.csv", index=False)
df_grouped.to_csv("GroupedResponses.csv", index=False)

print("✅ Combined dataset saved: Combined_dataset.json & GroupedResponses.csv")

✅ Combined dataset saved: Combined_dataset.json & GroupedResponses.csv


In [6]:
df_grouped = pd.read_csv("GroupedResponses.csv")
df_grouped.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7410 entries, 0 to 7409
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Context   7410 non-null   object
 1   Response  7410 non-null   object
 2   doc_id    7410 non-null   int64 
dtypes: int64(1), object(2)
memory usage: 173.8+ KB


# Set the embeddings and the BM25 just once

In [ ]:
from tqdm import tqdm
tqdm.pandas()  # enable .progress_apply

# BM25 on Questions only (Context)
documents = df_grouped["Context"].tolist()
tokenized = [
    [t for t in word_tokenize(doc.lower())
     if t not in STOPWORDS and t not in string.punctuation]
    for doc in tqdm(documents, desc="Tokenizing for BM25")
]
bm25 = BM25Okapi(tokenized)

texts = [
    f"{row['Context']}"
    for _, row in df_grouped.iterrows()
]

df_grouped["embedding"] = embed_texts(texts, batch_size=64).tolist()

# Save for later use
df_grouped.to_csv("GroupedResponsesWithEmbeddings.csv", index=False)

#FAISS index
index, id_to_meta = load_faiss_db(df_grouped, mode=FAISS_MODE)


Embedding batches: 100%|██████████| 116/116 [03:18<00:00,  1.71s/it]


# Load FAISS index, if you have saved first

In [ ]:
df_grouped = pd.read_csv('GroupedResponsesWithEmbeddings.csv')
documents = df_grouped["Context"].tolist()
tokenized = [
    [t for t in word_tokenize(doc.lower())
     if t not in STOPWORDS and t not in string.punctuation]
    for doc in tqdm(documents, desc="Tokenizing for BM25")
]
bm25 = BM25Okapi(tokenized)
index, id_to_meta = load_faiss_db(df_grouped, mode=FAISS_MODE)

Tokenizing for BM25: 100%|██████████| 7410/7410 [00:01<00:00, 6144.75it/s]


# Retrieval function (BM25 + FAISS fusion) -> top dicts

In [ ]:
def retrieve_top_docs(user_query: str, top_k: int = TOP_K, faiss_weight: float = ALPHA):
    doc_ids, fused = FuseScores(user_query, bm25, index, mode=FAISS_MODE, alpha=faiss_weight, stop_set=STOPWORDS)
    if len(doc_ids) == 0:
        return []
    top_idx = np.argsort(fused)[-top_k:][::-1]
    out = []
    for idx in top_idx:
        doc_id = doc_ids[idx]
        meta = id_to_meta[doc_id]
        out.append({"Context": meta["Context"], "Response": meta["Response"], "Score": float(fused[idx])})
    return out

# Example run (you can replace with your own query)

In [7]:
user_query = "I've been having suicidal thoughts at night, sometimes even in the middle of the day. What should I do?"
stepback_question = generate_stepback_question(user_query)

top_user = retrieve_top_docs(user_query, top_k=TOP_K)
top_stepback = retrieve_top_docs(stepback_question, top_k=TOP_K)

seen = set()
combined = []
for d in top_user + top_stepback:
    ctx = d["Context"]
    if ctx not in seen:
        seen.add(ctx)
        combined.append(d)

final_docs = rerank_with_cohere(user_query, combined, top_k=RERANK_TOP_K)

reply = generate_single_response(user_query, client, final_docs)

print("Retrieved docs (after rerank):", len(final_docs))
for i, d in enumerate(final_docs, 1):
    print(f"[{i}] Context: {d['Context']} Response: {d['Response']}")
print("\nAssistant:\n", reply, "\n")


Retrieved docs (after rerank): 5
[1] Context: I'm feeling really hopeless and suicidal. What should I do? Response: - It's important to take suicidal thoughts seriously and seek immediate help. Let's explore therapy and medication options to address your symptoms. We can also work on developing a safety plan and identifying supportive resources such as crisis lines and emergency services.
[2] Context: I've been having negative thoughts and feelings about myself and others, and sometimes I even think about hurting myself. What should I do? Response: - It's important to take thoughts of self-harm seriously and to seek immediate help if you ever feel like you're in danger of hurting yourself. We can work on finding treatment options that may include therapy, medication, or a combination of both. It's important to also have a support system in place and to avoid isolating yourself. You are not alone in this and there are people who care about you and want to help.
[3] Context: I'm really s

# Testing Set Generation

### Generate topic for each row in the dataset

In [1]:
import pandas as pd
from openai import OpenAI
from tqdm import tqdm
import os
import time
from dotenv import load_dotenv

# List of possible topics
topic_labels = [
    "Addiction",
    "Anger management",
    "Anxiety",
    "Behavioral issues",
    "Bipolar disorder",
    "Children and adolescent therapy",
    "Chronic pain and illness",
    "Codependency",
    "Couples therapy and relationship counseling",
    "Depression",
    "Domestic violence",
    "Eating disorders",
    "Family conflict",
    "Forgiveness",
    "Grief and loss",
    "Intimacy and sex",
    "LGBTQ+ issues",
    "Marriage counseling",
    "Obsessive-Compulsive Disorder (OCD)",
    "Parenting",
    "Self-esteem and self-worth",
    "Sleep disorders",
    "Social relationships",
    "Stress management",
    "Substance use disorders",
    "Trauma and PTSD",
    "Women's issues",
    "Workplace and career stress",
    "Attention-Deficit/Hyperactivity Disorder (ADHD)",
    "Autism Spectrum Disorder (ASD)"
]

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Load dataset
df = pd.read_csv("GroupedResponses.csv")

batch_size = 20
predicted_topics = []

for start in tqdm(range(0, len(df), batch_size)):
    batch_df = df.iloc[start:start+batch_size]

    for i, row in batch_df.iterrows():
        context = str(row["Context"])
        responses = str(row["Response"])  # still one string even if "-a -b -c"

        text = f"""
Context: {context}
Responses: {responses}

Classify this conversation into ONE main topic only from the following list:
{topic_labels}

Return ONLY the topic name exactly as it appears in the list.
"""

        topic = None
        attempts = 0
        while attempts < 3 and topic is None:
            try:
                completion = client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=[
                        {"role": "system", "content": "You are a precise classifier. Output exactly one topic name from the given list."},
                        {"role": "user", "content": text}
                    ],
                    temperature=0,
                    max_tokens=20
                )
                topic = completion.choices[0].message.content.strip()
            except Exception as e:
                print(f"⚠️ Error on row {i}, attempt {attempts+1}: {e}")
                attempts += 1
                time.sleep(2)

        predicted_topics.append(topic if topic else None)

# Save results
df["Topic"] = predicted_topics
df.to_csv("GroupedResponsesTopics.csv", index=False)
print("✅ Saved GroupedResponsesTopics.csv with topics")


100%|██████████| 371/371 [1:41:56<00:00, 16.49s/it]

✅ Saved GroupedResponsesTopics.csv with topics


### Making JSON for test set

In [8]:
import json
import pandas as pd

df = pd.read_csv("GroupedResponsesTopics_corrected.csv", sep = ",")

num_topics = df["Topic"].nunique()
print(f"Number of unique topics: {num_topics}")

topic_counts = df["Topic"].value_counts()
print(topic_counts)

Number of unique topics: 33
Topic
Stress management                                  983
Anxiety                                            861
Couples therapy and relationship counseling        854
Self-esteem and self-worth                         782
Workplace and career stress                        569
Depression                                         533
Social relationships                               328
Sleep disorders                                    312
Anger management                                   311
Grief and loss                                     278
Substance use disorders                            253
LGBTQ+ issues                                      206
Family conflict                                    196
Eating disorders                                   137
Parenting                                          133
Trauma and PTSD                                    118
Forgiveness                                        115
Attention-Deficit/Hyperactivity

### Generate base JSON with user_query and references

In [11]:
import pandas as pd
import json
from tqdm import tqdm
from openai import OpenAI

client = OpenAI()

selected_topics = [
    "Couples therapy and relationship counseling",
    "Anxiety",
    "Family conflict",
    "Depression",
    "Self-esteem and self-worth",
    "Trauma and PTSD",
    "Grief and loss",
    "Social relationships",
    "Domestic violence",
    "Anger management",
    "Workplace and career stress",
    "Stress management",
    "Forgiveness"
]

num_long_questions = 1 # how many short-medium test questions you want per topic
num_short_medium_questions = 1 # how many long test questions you want per topic

def summarize_response(responses, topic):
    """
    Summarize multiple therapist responses into a short coherent reference.
    """
    joined_text = " ".join(responses)
    prompt = (
        f"These are therapist responses related to {topic}. "
        "Summarize the main advice and insights in 4–6 sentences, "
        "keeping it factual, neutral, and concise."
        "Don't mention anything about the therapists.\n\n"
        f"Responses:\n{joined_text}"
    )
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7
    )
    return response.choices[0].message.content.strip()

test_data = []

for topic in tqdm(selected_topics, desc="Building test set"):
    subset = df[df["Topic"] == topic]
    if subset.empty:
        continue

    # Shuffle to randomize doc_id usage
    subset = subset.sample(frac=1, random_state=42).reset_index(drop=True)
    used_doc_ids = set()

    for i in range(num_short_medium_questions):
        # Filter out docs that were already used in this topic
        available_rows = subset[~subset["doc_id"].isin(used_doc_ids)] if "doc_id" in subset.columns else subset
        if available_rows.empty:
            break  # No more unique doc_ids left
        available_rows_short_medium = available_rows[(available_rows['Context'].str.len() < 150) & (available_rows['Context'].str.len() > 50)]

        # Pick 2 responses for summarization per question
        n_samples = min(2, len(available_rows_short_medium))
        sampled_rows_short_medium = available_rows_short_medium.sample(n=n_samples, random_state=42+i)

        user_input = sampled_rows_short_medium.iloc[0]["Context"]
        responses = sampled_rows_short_medium["Response"].tolist()
        reference = summarize_response(responses, topic)

        doc_ids = sampled_rows_short_medium["doc_id"].tolist() if "doc_id" in sampled_rows_short_medium.columns else None
        if doc_ids:
            used_doc_ids.update(doc_ids)

        test_data.append({
            "user_input": user_input,
            "reference": reference,
            "topic": topic,
            "doc_id": doc_ids
        })
    for j in range(num_long_questions):
        # Filter out docs that were already used in this topic
        available_rows = subset[~subset["doc_id"].isin(used_doc_ids)] if "doc_id" in subset.columns else subset
        if available_rows.empty:
            break  # No more unique doc_ids left
        available_rows_long = available_rows[available_rows['Context'].str.len() >= 200]

        # Pick 2 responses for summarization per question
        n_samples = min(2, len(available_rows_long))
        sampled_rows_long = available_rows_long.sample(n=n_samples, random_state=42+i)

        user_input = sampled_rows_long.iloc[0]["Context"]
        responses = sampled_rows_long["Response"].tolist()
        reference = summarize_response(responses, topic)

        doc_ids = sampled_rows_long["doc_id"].tolist() if "doc_id" in sampled_rows_long.columns else None
        if doc_ids:
            used_doc_ids.update(doc_ids)

        test_data.append({
            "user_input": user_input,
            "reference": reference,
            "topic": topic,
            "doc_id": doc_ids
        })


with open("new_expanded_ragas_testset.json", "w", encoding="utf-8") as f:
    json.dump(test_data, f, indent=2, ensure_ascii=False)

print(f"✅ Done! Saved {len(test_data)} test entries to new_expanded_ragas_testset.json")


Building test set: 100%|██████████| 13/13 [01:06<00:00,  5.15s/it]

✅ Done! Saved 26 test entries to new_expanded_ragas_testset.json


### Make JSON for Advanced RAG evaluation

In [17]:
import os
import json
from tqdm import tqdm

def run_rag_fusion_sweep():
    # === Load your base JSON ===
    with open("new_expanded_ragas_testset.json", "r", encoding="utf-8") as f:
        base_data = json.load(f)

    # === Sweep through FAISS weights 0.1–0.9 ===
    for i, alpha in enumerate([round(x, 1) for x in [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]], start=1):
        folder_name = f"{i:02d}"  # e.g., 01, 02, 03
        save_dir = os.path.join("new_expanded_fusion_results", folder_name)
        os.makedirs(save_dir, exist_ok=True)

        save_path = os.path.join(save_dir, f"rag_evaluation_set_{folder_name}.json")

        # === Skip if file already exists ===
        if os.path.exists(save_path):
            print(f"⏭️ Skipping FAISS weight={alpha} ({folder_name}/) — already exists.")
            continue

        print(f"\n=== Running fusion with FAISS weight = {alpha} ({folder_name}/) ===")
        augmented_data = []

        for item in tqdm(base_data, desc=f"Weight={alpha}"):
            user_query = item["user_input"]

            try:
                # === Step 1: Generate stepback question ===
                stepback_question = generate_stepback_question(user_query)

                # === Step 2: Retrieve documents (with current faiss_weight) ===
                top_user = retrieve_top_docs(user_query, top_k=TOP_K, faiss_weight=alpha)
                top_stepback = retrieve_top_docs(stepback_question, top_k=TOP_K, faiss_weight=alpha)

                # === Step 3: Combine & deduplicate ===
                seen = set()
                combined = []
                for d in top_user + top_stepback:
                    ctx = d["Context"]
                    if ctx not in seen:
                        seen.add(ctx)
                        combined.append(d)

                # === Step 4: Rerank ===
                final_docs = rerank_with_cohere(user_query, combined, top_k=RERANK_TOP_K)

                # === Step 5: Generate model response ===
                reply = generate_single_response(user_query, client, final_docs)

                # === Step 6: Build output item ===
                augmented_item = {
                    "user_input": user_query,
                    "reference": item["reference"],
                    "topic": item["topic"],
                    "doc_id": item["doc_id"],
                    "retrieved_contexts": [d["Response"] for d in final_docs],
                    "response": reply,
                    "faiss_weight": alpha
                }

            except Exception as e:
                print(f"⚠️ Error processing '{user_query[:40]}...': {e}")
                augmented_item = {
                    "user_input": user_query,
                    "reference": item["reference"],
                    "topic": item["topic"],
                    "doc_id": item["doc_id"],
                    "retrieved_contexts": [],
                    "response": None,
                    "error": str(e),
                    "faiss_weight": alpha
                }

            augmented_data.append(augmented_item)

        # === Save JSON for this alpha ===
        with open(save_path, "w", encoding="utf-8") as f:
            json.dump(augmented_data, f, indent=2, ensure_ascii=False)

        print(f"✅ Saved {len(augmented_data)} samples to {save_path}")

# === Run the sweep ===
run_rag_fusion_sweep()


⏭️ Skipping FAISS weight=0.1 (01/) — already exists.
⏭️ Skipping FAISS weight=0.2 (02/) — already exists.
⏭️ Skipping FAISS weight=0.3 (03/) — already exists.

=== Running fusion with FAISS weight = 0.4 (04/) ===


Weight=0.4: 100%|██████████| 26/26 [04:37<00:00, 10.69s/it]

✅ Saved 26 samples to new_expanded_fusion_results\04\rag_evaluation_set_04.json
⏭️ Skipping FAISS weight=0.5 (05/) — already exists.
⏭️ Skipping FAISS weight=0.6 (06/) — already exists.
⏭️ Skipping FAISS weight=0.7 (07/) — already exists.
⏭️ Skipping FAISS weight=0.8 (08/) — already exists.
⏭️ Skipping FAISS weight=0.9 (09/) — already exists.


### Functions for naive RAG

In [14]:
def naive_retrieve_top_docs(user_query: str, top_k: int = 5):
    """Naïve retrieval: only semantic (FAISS) or only BM25, no fusion."""
    # Option 1: Semantic only (FAISS)
    q_emb = np.array([get_embedding(user_query)], dtype=np.float32)
    faiss.normalize_L2(q_emb)
    D, I = index.search(q_emb, k=top_k)
    docs = [{"Context": id_to_meta[i]["Context"],
             "Response": id_to_meta[i]["Response"],
             "Score": float(D[0][j])}
            for j, i in enumerate(I[0])]
    return docs

def naive_retrieve_top_docs_bm25(user_query: str, top_k: int = 5):
    """Naïve retrieval using only BM25 (no fusion, no semantic search)."""
    query_tokens = user_query.lower().split()
    scores = bm25.get_scores(query_tokens)
    top_indices = np.argsort(scores)[::-1][:top_k]  # descending order
    
    docs = [
        {
            "Context": id_to_meta[i]["Context"],
            "Response": id_to_meta[i]["Response"],
            "Score": float(scores[i]),
        }
        for i in top_indices
    ]
    return docs

def naive_rag_generate(user_query: str, top_k: int = 5):
    """Full naïve RAG pipeline: no stepback, no rerank, no fusion."""
    retrieved_docs = naive_retrieve_top_docs(user_query, top_k=top_k)
    reply = generate_single_response(user_query, client, retrieved_docs)
    return {
        "user_input": user_query,
        "retrieved_contexts": [d["Response"] for d in retrieved_docs],
        "response": reply
    }

def naive_rag_generate_bm25(user_query: str, top_k: int = 5):
    """Full naïve RAG pipeline: no stepback, no rerank, no fusion."""
    retrieved_docs = naive_retrieve_top_docs_bm25(user_query, top_k=top_k)
    reply = generate_single_response(user_query, client, retrieved_docs)
    return {
        "user_input": user_query,
        "retrieved_contexts": [d["Response"] for d in retrieved_docs],
        "response": reply
    }

### Generate json for naive RAG with FAISS

In [ ]:
import json
import numpy as np
from tqdm import tqdm

# === Load base test JSON ===
with open("new_expanded_ragas_testset.json", "r", encoding="utf-8") as f:
    base_data = json.load(f)

augmented_data = []

# === Loop through each query ===
for item in tqdm(base_data, desc="Running Naïve RAG"):
    user_query = item["user_input"]

    try:
        # Run naïve RAG generation
        rag_output = naive_rag_generate(user_query, top_k=TOP_K)

        # Merge with original reference info
        augmented_item = {
            "user_input": user_query,
            "reference": item.get("reference"),
            "topic": item.get("topic"),
            "doc_id": item.get("doc_id"),
            "retrieved_contexts": rag_output["retrieved_contexts"],
            "response": rag_output["response"]
        }

    except Exception as e:
        print(f"⚠️ Error processing query '{user_query[:40]}...': {e}")
        augmented_item = {
            "user_input": user_query,
            "reference": item.get("reference"),
            "topic": item.get("topic"),
            "doc_id": item.get("doc_id"),
            "retrieved_contexts": [],
            "response": None,
            "error": str(e)
        }

    augmented_data.append(augmented_item)


# === Save final JSON ===
with open("new_expanded_naive_rag_evaluation_set_faiss.json", "w", encoding="utf-8") as f:
    json.dump(augmented_data, f, indent=2, ensure_ascii=False)

print(f"✅ Done! Saved {len(augmented_data)} entries to new_expanded_naive_rag_evaluation_set_faiss.json")


Running Naïve RAG: 100%|██████████| 26/26 [02:54<00:00,  6.73s/it]

✅ Done! Saved 26 entries to new_expanded_naive_rag_evaluation_set.json


### Generate json for naive RAG with BM25

In [18]:
import json
import numpy as np
from tqdm import tqdm


# === Load base test JSON ===
with open("new_expanded_ragas_testset.json", "r", encoding="utf-8") as f:
    base_data = json.load(f)

augmented_data = []

# === Loop through each query ===
for item in tqdm(base_data, desc="Running Naïve RAG"):
    user_query = item["user_input"]

    try:
        # Run naïve RAG generation
        rag_output = naive_rag_generate_bm25(user_query, top_k=TOP_K)

        # Merge with original reference info
        augmented_item = {
            "user_input": user_query,
            "reference": item.get("reference"),
            "topic": item.get("topic"),
            "doc_id": item.get("doc_id"),
            "retrieved_contexts": rag_output["retrieved_contexts"],
            "response": rag_output["response"]
        }

    except Exception as e:
        print(f"⚠️ Error processing query '{user_query[:40]}...': {e}")
        augmented_item = {
            "user_input": user_query,
            "reference": item.get("reference"),
            "topic": item.get("topic"),
            "doc_id": item.get("doc_id"),
            "retrieved_contexts": [],
            "response": None,
            "error": str(e)
        }

    augmented_data.append(augmented_item)


# === Save final JSON ===
with open("new_expanded_naive_rag_evaluation_set_bm25.json", "w", encoding="utf-8") as f:
    json.dump(augmented_data, f, indent=2, ensure_ascii=False)

print(f"✅ Done! Saved {len(augmented_data)} entries to new_expanded_naive_rag_evaluation_set_bm25.json")


Running Naïve RAG: 100%|██████████| 26/26 [02:58<00:00,  6.85s/it]

✅ Done! Saved 26 entries to new_expanded_naive_rag_evaluation_set_bm25.json


# Function for ablation study

In [19]:
def run_rag_pipeline(user_query: str,
                     use_stepback: bool = True,
                     use_hybrid: bool = True,
                     use_rerank: bool = True,
                     retriever_mode: str = "hybrid",
                     top_k: int = TOP_K,
                     rerank_top_k: int = RERANK_TOP_K):
    """
    Unified RAG pipeline with adaptive retrieval depth for ablation testing.
    If stepback is disabled, we automatically expand top_k to give reranker more candidates.
    """

    # --- Adaptive retrieval depth ---
    if not use_stepback:
        adaptive_top_k = max(top_k * 3, 15)
    else:
        adaptive_top_k = top_k

    # 1️⃣ Stepback question
    if use_stepback:
        stepback_question = generate_stepback_question(user_query)
        stepback_docs = retrieve_top_docs(stepback_question, top_k=top_k)
    else:
        stepback_question = None
        stepback_docs = []

    # 2️⃣ Retrieval mode logic
    if use_hybrid or retriever_mode == "hybrid":
        top_user = retrieve_top_docs(user_query, top_k=adaptive_top_k)

    elif retriever_mode == "faiss":
        q_emb = np.array([get_embedding(user_query)], dtype=np.float32)
        faiss.normalize_L2(q_emb)
        D, I = index.search(q_emb, k=adaptive_top_k)
        top_user = [{
            "Context": id_to_meta[i]["Context"],
            "Response": id_to_meta[i]["Response"],
            "Score": float(D[0][j])
        } for j, i in enumerate(I[0])]

    elif retriever_mode == "bm25":
        tokens = user_query.lower().split()
        scores = bm25.get_scores(tokens)
        top_idx = np.argsort(scores)[::-1][:adaptive_top_k]
        top_user = [{
            "Context": id_to_meta[i]["Context"],
            "Response": id_to_meta[i]["Response"],
            "Score": float(scores[i])
        } for i in top_idx]

    else:
        raise ValueError(f"Invalid retriever_mode: {retriever_mode}")

    # 3️⃣ Combine (deduplicate)
    seen = set()
    combined = []
    for d in top_user + stepback_docs:
        ctx = d["Context"]
        if ctx not in seen:
            seen.add(ctx)
            combined.append(d)

    # 4️⃣ Optional rerank
    if use_rerank:
        final_docs = rerank_with_cohere(user_query, combined, top_k=rerank_top_k)
    else:
        final_docs = combined[:rerank_top_k]

    # 5️⃣ Generate response
    reply = generate_single_response(user_query, client, final_docs)

    return {
        "user_input": user_query,
        "stepback_used": use_stepback,
        "hybrid_used": use_hybrid,
        "rerank_used": use_rerank,
        "retriever_mode": retriever_mode,
        "adaptive_top_k": adaptive_top_k,
        "retrieved_contexts": [d["Response"] for d in final_docs],
        "response": reply,
    }


In [21]:
import json
from tqdm import tqdm

# === Load test queries ===
with open("new_expanded_ragas_testset.json", "r", encoding="utf-8") as f:
    base_data = json.load(f)

def run_ablation_eval(config_name, **kwargs):
    augmented_data = []
    for item in tqdm(base_data, desc=f"Running {config_name}"):
        user_query = item["user_input"]
        try:
            rag_output = run_rag_pipeline(user_query, **kwargs)
            augmented_item = {
                "user_input": user_query,
                "reference": item.get("reference"),
                "topic": item.get("topic"),
                "doc_id": item.get("doc_id"),
                "retrieved_contexts": rag_output["retrieved_contexts"],
                "response": rag_output["response"],
            }
        except Exception as e:
            print(f"⚠️ Error on query '{user_query[:40]}...': {e}")
            augmented_item = {
                "user_input": user_query,
                "reference": item.get("reference"),
                "topic": item.get("topic"),
                "doc_id": item.get("doc_id"),
                "retrieved_contexts": [],
                "response": None,
                "error": str(e),
            }
        augmented_data.append(augmented_item)

    out_name = f"new_expanded_rag_ablation_{config_name}.json"
    with open(out_name, "w", encoding="utf-8") as f:
        json.dump(augmented_data, f, indent=2, ensure_ascii=False)
    print(f"✅ Saved {len(augmented_data)} entries to {out_name}")


# === Run 4 ablation configurations ===
run_ablation_eval("no_stepback_hybrid_rerank",
                  use_stepback=False, use_hybrid=True, use_rerank=True, retriever_mode="hybrid")



Running no_stepback_hybrid_rerank: 100%|██████████| 26/26 [03:58<00:00,  9.16s/it]

✅ Saved 26 entries to new_expanded_rag_ablation_no_stepback_hybrid_rerank.json


In [22]:
run_ablation_eval("no_hybrid_faiss_rerank_stepback",
                  use_stepback=True, use_hybrid=False, use_rerank=True, retriever_mode="faiss")

run_ablation_eval("no_hybrid_bm25_rerank_stepback",
                  use_stepback=True, use_hybrid=False, use_rerank=True, retriever_mode="bm25")

run_ablation_eval("no_rerank_hybrid_stepback",
                  use_stepback=True, use_hybrid=True, use_rerank=False, retriever_mode="hybrid")

Running no_hybrid_faiss_rerank_stepback: 100%|██████████| 26/26 [04:16<00:00,  9.85s/it]


✅ Saved 26 entries to new_expanded_rag_ablation_no_hybrid_faiss_rerank_stepback.json


Running no_hybrid_bm25_rerank_stepback: 100%|██████████| 26/26 [03:50<00:00,  8.87s/it]


✅ Saved 26 entries to new_expanded_rag_ablation_no_hybrid_bm25_rerank_stepback.json


Running no_rerank_hybrid_stepback: 100%|██████████| 26/26 [03:45<00:00,  8.67s/it]

✅ Saved 26 entries to new_expanded_rag_ablation_no_rerank_hybrid_stepback.json
